In [153]:
from pathlib import Path
import pandas as pd
import re

In [154]:
base_path = Path("../data/bronze/laps/season=2026")

In [155]:
files = sorted(base_path.rglob("*.parquet"))
print(f"Found {len(files)} files")
for f in files:
    print(f"- {f}")

Found 3 files
- ..\data\bronze\laps\season=2026\round=01\laps.parquet
- ..\data\bronze\laps\season=2026\round=02\laps.parquet
- ..\data\bronze\laps\season=2026\round=03\laps.parquet


In [156]:
dfs = [pd.read_parquet(f) for f in files]
df = pd.concat(dfs, ignore_index=True)

In [157]:
df_silver = df.drop(columns=["FastF1Generated", "ingested_at_utc", "IsAccurate"])

In [158]:
key_cols = ["season", "round_number", "Driver", "LapNumber"]
dup_cnt = df_silver.duplicated(subset=key_cols).sum()
print(f"Duplicated by key: {dup_cnt}")

Duplicated by key: 0


In [159]:
df.to_csv("./cek_isi.csv", index=False)

**FUNCTION**

In [160]:
def check_null(column):
    amount = column.isna().sum()
    print(f"Null data for {column.name}: {amount}")
    if amount > 0:
        mask = column.isna()
        if mask.any():
            print(f"Rows with null in {column.name}:")
            display(df.loc[mask])
def strip_string(column):
    print(f"Stripping string for {column.name}")
    column = column.str.strip()
    
    return column

def upper_string(column):
    print(f"Uppercasing string for {column.name}")
    column = column.str.upper()

    return column

def clean_driver(column):
    column = strip_string(column)
    column = upper_string(column)

    return column

def convert_to_int(column):
    print(f"Converting to int for {column.name}")
    column = pd.to_numeric(column, errors="coerce")
    
    return column

**TIME**

In [161]:
check_null(df_silver["Time"])

Null data for Time: 0


**DRIVER**

In [162]:
check_null(df_silver["Driver"])
df_silver.value_counts("Driver")
df_silver["Driver"] = clean_driver(df_silver["Driver"])

Null data for Driver: 0
Stripping string for Driver
Uppercasing string for Driver


**DRIVER NUMBER**

In [163]:
check_null(df_silver["DriverNumber"])
df_silver["DriverNumber"] = convert_to_int(df_silver["DriverNumber"])

Null data for DriverNumber: 0
Converting to int for DriverNumber


**LAP TIME**

In [164]:
check_null(df_silver["LapTime"])
df_silver["is_LapTime_not_na"] = df_silver["LapTime"].notna()

Null data for LapTime: 48
Rows with null in LapTime:


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,season,round_number,event_name,session_type,ingested_at_utc
240,0 days 01:38:59.952000,ALO,14,NaT,13.0,2.0,NaT,0 days 01:22:49.657000,0 days 00:00:35.885000,0 days 00:00:25.378000,...,19.0,None,,False,False,2026,1,Australian Grand Prix,R,2026-04-27T11:14:12.581298+00:00
249,0 days 01:52:27.116000,ALO,14,NaT,22.0,3.0,NaT,0 days 01:52:27.205000,0 days 00:00:30.824000,0 days 00:00:30.521000,...,18.0,None,,False,False,2026,1,Australian Grand Prix,R,2026-04-27T11:14:12.581298+00:00
341,0 days 02:13:18.247000,STR,18,NaT,34.0,3.0,NaT,0 days 01:55:18.611000,0 days 00:00:30.496000,0 days 00:00:19.613000,...,17.0,None,,False,False,2026,1,Australian Grand Prix,R,2026-04-27T11:14:12.581298+00:00
408,0 days 01:03:51.908000,HUL,27,NaT,1.0,1.0,NaT,NaT,NaT,NaT,...,NaN,None,,True,False,2026,1,Australian Grand Prix,R,2026-04-27T11:14:12.581298+00:00
875,0 days 01:19:15.504000,HAD,6,NaT,11.0,1.0,NaT,NaT,NaT,NaT,...,NaN,None,None,True,False,2026,1,Australian Grand Prix,R,2026-04-27T11:14:12.581298+00:00
945,0 days 01:22:05.068000,BOT,77,NaT,12.0,1.0,NaT,0 days 01:21:36.868000,0 days 00:00:46.720000,0 days 00:00:45.988000,...,19.0,None,,False,False,2026,1,Australian Grand Prix,R,2026-04-27T11:14:12.581298+00:00
949,0 days 01:29:21.898000,BOT,77,NaT,16.0,2.0,NaT,NaT,NaT,NaT,...,NaN,None,None,True,False,2026,1,Australian Grand Prix,R,2026-04-27T11:14:12.581298+00:00
1007,0 days 00:58:03.760000,NOR,1,NaT,1.0,1.0,NaT,NaT,NaT,NaT,...,NaN,None,,True,False,2026,2,Chinese Grand Prix,R,2026-04-27T11:13:56.741863+00:00
1018,0 days 01:15:47.651000,GAS,10,NaT,11.0,2.0,0 days 01:13:30.471000,NaT,0 days 00:00:54.680000,0 days 00:00:40.781000,...,9.0,None,,False,False,2026,2,Chinese Grand Prix,R,2026-04-27T11:13:56.741863+00:00
1019,0 days 01:18:17.634000,GAS,10,NaT,12.0,2.0,NaT,NaT,0 days 00:00:42.739000,0 days 00:00:44.215000,...,9.0,None,,False,False,2026,2,Chinese Grand Prix,R,2026-04-27T11:13:56.741863+00:00


In [165]:
df_silver.info()
df_silver

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3038 entries, 0 to 3037
Data columns (total 34 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   Time                3038 non-null   timedelta64[ns]
 1   Driver              3038 non-null   object         
 2   DriverNumber        3038 non-null   int64          
 3   LapTime             2990 non-null   timedelta64[ns]
 4   LapNumber           3038 non-null   float64        
 5   Stint               3038 non-null   float64        
 6   PitOutTime          80 non-null     timedelta64[ns]
 7   PitInTime           84 non-null     timedelta64[ns]
 8   Sector1Time         2968 non-null   timedelta64[ns]
 9   Sector2Time         3029 non-null   timedelta64[ns]
 10  Sector3Time         3026 non-null   timedelta64[ns]
 11  Sector1SessionTime  2957 non-null   timedelta64[ns]
 12  Sector2SessionTime  3029 non-null   timedelta64[ns]
 13  Sector3SessionTime  3026 non-null

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,season,round_number,event_name,session_type,is_LapTime_not_na
0,0 days 01:03:56.437000,NOR,1,0 days 00:01:36.458000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:18.163000,...,NaT,1,6.0,None,,2026,1,Australian Grand Prix,R,True
1,0 days 01:05:23.781000,NOR,1,0 days 00:01:27.344000,2.0,1.0,NaT,NaT,0 days 00:00:31.074000,0 days 00:00:18.116000,...,NaT,1,6.0,None,,2026,1,Australian Grand Prix,R,True
2,0 days 01:06:50.644000,NOR,1,0 days 00:01:26.863000,3.0,1.0,NaT,NaT,0 days 00:00:30.541000,0 days 00:00:18.252000,...,NaT,1,7.0,None,,2026,1,Australian Grand Prix,R,True
3,0 days 01:08:16.501000,NOR,1,0 days 00:01:25.857000,4.0,1.0,NaT,NaT,0 days 00:00:30.190000,0 days 00:00:18.193000,...,NaT,1,7.0,None,,2026,1,Australian Grand Prix,R,True
4,0 days 01:09:42.074000,NOR,1,0 days 00:01:25.573000,5.0,1.0,NaT,NaT,0 days 00:00:29.930000,0 days 00:00:17.868000,...,NaT,1,7.0,None,,2026,1,Australian Grand Prix,R,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3033,0 days 01:31:41.771000,BEA,87,0 days 00:01:57.448000,17.0,2.0,0 days 01:30:08.107000,NaT,0 days 00:00:57.139000,0 days 00:00:41.840000,...,NaT,1,19.0,None,,2026,3,Japanese Grand Prix,R,True
3034,0 days 01:33:17.825000,BEA,87,0 days 00:01:36.054000,18.0,2.0,NaT,NaT,0 days 00:00:35.006000,0 days 00:00:42.424000,...,NaT,1,19.0,None,,2026,3,Japanese Grand Prix,R,True
3035,0 days 01:34:53.730000,BEA,87,0 days 00:01:35.905000,19.0,2.0,NaT,NaT,0 days 00:00:35.345000,0 days 00:00:42.252000,...,NaT,1,18.0,None,,2026,3,Japanese Grand Prix,R,True
3036,0 days 01:36:29.334000,BEA,87,0 days 00:01:35.604000,20.0,2.0,NaT,NaT,0 days 00:00:35.253000,0 days 00:00:42.310000,...,NaT,1,18.0,None,,2026,3,Japanese Grand Prix,R,True
